In [8]:
import random

COLORS, VALUES = ['Red', 'Blue', 'Green', 'Yellow'], [str(i) for i in range(10)] + ['Skip']

class Card:
    def __init__(self, color, value):
        self.color, self.value = color, value
    def __repr__(self): return f"{self.color[0]}{self.value}"
    def __eq__(self, other): return isinstance(other, Card) and self.color == other.color and self.value == other.value
    def __hash__(self): return hash((self.color, self.value))

class GameState:
    def __init__(self, hands, top, deck, turn=0, consecutive_draws=0):
        self.hands, self.top, self.deck, self.turn = hands, top, deck, turn
        self.consecutive_draws = consecutive_draws

    def is_terminal(self):
        if any(len(h) == 0 for h in self.hands): return True
        if self.consecutive_draws >= len(self.hands) and len(self.deck) == 0: return True
        return False

    def clone(self):
        return GameState([list(h) for h in self.hands], self.top, list(self.deck), self.turn, self.consecutive_draws)

def get_valid_moves(hand, top):
    return [c for c in hand if c.color == top.color or c.value == top.value]

def apply_move(state, move):
    s = state.clone()
    p = s.turn
    if move == 'Draw':
        if s.deck: s.hands[p].append(s.deck.pop(0))
        s.consecutive_draws += 1
        s.turn = (p + 1) % 3
    else:
        s.hands[p].remove(move)
        s.top, s.consecutive_draws = move, 0
        s.turn = (p + (2 if move.value == 'Skip' else 1)) % 3
    return s

In [9]:
def evaluate(state, ai_idx):
    if len(state.hands[ai_idx]) == 0: return 100
    if any(len(h) == 0 for i, h in enumerate(state.hands) if i != ai_idx): return -100
    return sum(len(h) for i, h in enumerate(state.hands) if i != ai_idx) - len(state.hands[ai_idx])

def minimax(state, depth, ai_idx):
    if state.is_terminal() or depth == 0: return evaluate(state, ai_idx)
    moves = get_valid_moves(state.hands[state.turn], state.top)
    if not moves: return minimax(apply_move(state, 'Draw'), depth - 1, ai_idx)

    results = [minimax(apply_move(state, m), depth - 1, ai_idx) for m in moves]
    return max(results) if state.turn == ai_idx else min(results)

def get_best_move(state, depth, ai_idx):
    moves = get_valid_moves(state.hands[state.turn], state.top)
    if not moves: return 'Draw'
    return max(moves, key=lambda m: minimax(apply_move(state, m), depth - 1, ai_idx))

In [10]:
def setup_game():
    deck = [Card(c, v) for c in COLORS for v in VALUES]
    random.shuffle(deck)
    hands = [[deck.pop(0) for _ in range(5)] for _ in range(3)]
    top = deck.pop(0)
    while top.value == 'Skip': top = deck.pop(0)
    return GameState(hands, top, deck)

def play_game():
    mode = input("Enter mode (manual / simulation): ").strip().lower()
    state = setup_game()
    names = ["P1-AI", "P2-AI", "P3-YOU" if mode == "manual" else "P3-AI"]

    print(f"\nSTART | Top: {state.top} | Hands: {[len(h) for h in state.hands]}")
    print("-" * 50)

    while not state.is_terminal():
        p = state.turn
        if mode == "manual" and p == 2:
            # Human Turn
            moves = get_valid_moves(state.hands[p], state.top)
            print(f"\nYOUR HAND: {state.hands[p]} | TOP: {state.top}")
            if not moves:
                print("No moves! You must draw.")
                move = 'Draw'
            else:
                for i, m in enumerate(moves): print(f" {i}: {m}")
                idx = int(input("Pick index: "))
                move = moves[idx]
        else:
            # AI Turn
            move = get_best_move(state, 2, p)

        move_desc = f"played {move}" if move != 'Draw' else "DRAW"
        state = apply_move(state, move)
        print(f"[{names[p]}] {move_desc.ljust(11)} | Cards: {len(state.hands[p])} | Top: {state.top}")

    print("-" * 50)
    winner = next((i for i, h in enumerate(state.hands) if len(h) == 0), -1)
    print(f"GAME OVER | {'Winner: ' + names[winner] if winner != -1 else 'Stalemate'}")

play_game()


START | Top: G7 | Hands: [5, 5, 5]
--------------------------------------------------
[P1-AI] played R7   | Cards: 4 | Top: R7
[P2-AI] DRAW        | Cards: 6 | Top: R7
[P3-AI] DRAW        | Cards: 6 | Top: R7
[P1-AI] played R2   | Cards: 3 | Top: R2
[P2-AI] played R9   | Cards: 5 | Top: R9
[P3-AI] played G9   | Cards: 5 | Top: G9
[P1-AI] played G3   | Cards: 2 | Top: G3
[P2-AI] played GSkip | Cards: 4 | Top: GSkip
[P1-AI] DRAW        | Cards: 3 | Top: GSkip
[P2-AI] played G1   | Cards: 3 | Top: G1
[P3-AI] played G6   | Cards: 4 | Top: G6
[P1-AI] played B6   | Cards: 2 | Top: B6
[P2-AI] played BSkip | Cards: 2 | Top: BSkip
[P1-AI] DRAW        | Cards: 3 | Top: BSkip
[P2-AI] played B9   | Cards: 1 | Top: B9
[P3-AI] played B1   | Cards: 3 | Top: B1
[P1-AI] DRAW        | Cards: 4 | Top: B1
[P2-AI] played Y1   | Cards: 0 | Top: Y1
--------------------------------------------------
GAME OVER | Winner: P2-AI
